# AO3 Tag Visualizer

Notebook version of `ao3_tag_visualizer.py`. Reads an existing `ao3_tag_metadata.csv`
(produced by `ao3_tag_scraper.py`/`.ipynb`) and visualizes connections between the seed
tag (the popular tag from `archiveofourown.org/tags` used as the search term) and each
work's `rating`, `warnings`, `category`, `fandom`, and `additional_tags`:

- an **interactive bipartite network graph** (seed tags <-> attribute values, edges
  weighted by co-occurrence count)
- a **co-occurrence heatmap per field** (rows = seed tags, columns = attribute values)

`fandom` and `additional_tags` are high-cardinality, so each is filtered to its top-N
most frequent values (computed from the full dataset) before either visualization is
built.

**No network access is required** -- this only reads a local CSV.

The first code cell installs this notebook's dependencies (pandas, networkx, pyvis, matplotlib, seaborn) if they're missing -- safe to re-run.

Edit the Configuration cell, then run all cells in order.

**Kernel restart required after updating:** if you've previously run an older version
of this notebook in the same session, do **Kernel -> Restart Kernel and Run All Cells**.
Stale function definitions from a previous run persist in memory until the kernel is
restarted.

In [ ]:
# Installs any of this notebook's dependencies that aren't already present
# (pandas, networkx, pyvis, matplotlib, seaborn). Safe to re-run -- already
# satisfied requirements are a fast no-op. Restart the kernel afterward if
# this is the first time you've run this cell in the current session.
%pip install -q pandas networkx pyvis matplotlib seaborn

In [ ]:
import os
import re

import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
import seaborn as sns
from pyvis.network import Network

## Configuration

Edit these values, then run all cells in order.

In [ ]:
INPUT = "ao3_tag_metadata.csv"
TOP_FANDOMS = 20              # keep only the top N most frequent fandoms overall
TOP_ADDITIONAL_TAGS = 40      # keep only the top N most frequent additional tags overall
MIN_COUNT = 2                 # drop edges/cells below this co-occurrence count
TOP_SEED_TAGS = None          # limit to the N seed tags with the most works (None = all)
NETWORK_OUT = "ao3_tag_network.html"
HEATMAP_OUT_DIR = "heatmaps"
NORMALIZE = False             # heatmap cells show % of seed tag's works instead of raw counts

DELIMITER = ", "
FIELDS_TO_VISUALIZE = ["rating", "warnings", "category", "fandom", "additional_tags"]
FIELDS_TOP_N_ELIGIBLE = {"fandom", "additional_tags"}
FIELD_COLORS = {
    "seed_tag": "#4C72B0", "rating": "#DD8452", "warnings": "#55A868",
    "category": "#C44E52", "fandom": "#8172B2", "additional_tags": "#937860",
}

## Load and explode metadata

The CSV is loaded once. Multi-value fields (e.g. `additional_tags`, `category`) are joined with `", "` in the scraper's output, so they're split back out into one row per distinct value per work -- this is what makes a work with 3 additional tags contribute 3 separate co-occurrence increments, not 1.

In [ ]:
def load_metadata(input_csv):
    df = pd.read_csv(input_csv, dtype=str, keep_default_na=False)
    if df.empty:
        raise ValueError(f"{input_csv} has no data rows.")
    return df


def split_values(cell, delimiter=DELIMITER):
    if not cell:
        return []
    values = [v.strip() for v in cell.split(delimiter) if v.strip()]
    return list(dict.fromkeys(values))


def explode_field(df, field):
    exploded = df[["tag", "work_id", field]].copy()
    exploded[field] = exploded[field].map(split_values)
    exploded = exploded.explode(field)
    exploded = exploded[exploded[field].notna() & (exploded[field] != "")]
    return exploded


df = load_metadata(INPUT)
df.head()

## Co-occurrence counting and top-N filtering

`fandom` and `additional_tags` are filtered to their top-N most frequent values, computed from the *full* dataset before any per-seed-tag filtering -- so a kept value's count always reflects its true frequency, not a reduced one.

In [ ]:
def total_value_counts(exploded, field):
    return exploded[field].value_counts()


def top_n_values(exploded, field, n):
    if n is None:
        return None
    counts = total_value_counts(exploded, field).sort_values(ascending=False)
    counts = counts.reset_index()
    counts.columns = [field, "count"]
    counts = counts.sort_values(["count", field], ascending=[False, True])
    return set(counts[field].head(n))


def cooccurrence_counts(exploded, field, keep_values):
    if keep_values is not None:
        exploded = exploded[exploded[field].isin(keep_values)]
    if exploded.empty:
        return pd.DataFrame(columns=["tag", field, "count"])
    return exploded.groupby(["tag", field]).size().reset_index(name="count")


def apply_min_count(counts, min_count):
    return counts[counts["count"] >= min_count]


def rank_seed_tags(df, top_seed_tags):
    ranked = df["tag"].value_counts().index.tolist()
    return ranked if top_seed_tags is None else ranked[:top_seed_tags]


def build_field_data(df, top_fandoms, top_additional_tags, min_count):
    top_n_by_field = {"fandom": top_fandoms, "additional_tags": top_additional_tags}
    field_tables = {}
    for field in FIELDS_TO_VISUALIZE:
        exploded = explode_field(df, field)
        n = top_n_by_field.get(field) if field in FIELDS_TOP_N_ELIGIBLE else None
        keep_values = top_n_values(exploded, field, n)
        counts = cooccurrence_counts(exploded, field, keep_values)
        counts = apply_min_count(counts, min_count)
        field_tables[field] = counts
    return field_tables


seed_tags = rank_seed_tags(df, TOP_SEED_TAGS)
field_tables = build_field_data(df, TOP_FANDOMS, TOP_ADDITIONAL_TAGS, MIN_COUNT)
{field: len(counts) for field, counts in field_tables.items()}  # preview: surviving edges per field

## Interactive network graph

Nodes: seed tags (blue) + attribute values (one color per field). Edges: co-occurrence count (hover for the exact count). Pan/zoom/drag are built in; no internet connection is needed to view the resulting HTML file.

In [ ]:
def build_bipartite_graph(field_tables, seed_tags):
    graph = nx.Graph()
    seed_tag_set = set(seed_tags)
    for tag in seed_tags:
        graph.add_node(f"tag::{tag}", label=tag, group="seed_tag",
                        color=FIELD_COLORS["seed_tag"], title=tag)

    for field, counts in field_tables.items():
        for _, row in counts.iterrows():
            tag, value, count = row["tag"], row[field], row["count"]
            if tag not in seed_tag_set:
                continue
            value_id = f"{field}::{value}"
            if value_id not in graph:
                graph.add_node(value_id, label=value, group=field,
                                color=FIELD_COLORS[field], title=f"{field}: {value}")
            graph.add_edge(f"tag::{tag}", value_id, weight=int(count),
                            title=f"{tag} × {value}: {count} works")
    return graph


_BOOTSTRAP_TAG_RE = re.compile(
    r'<(?:link|script)[^>]*\bhttps://cdn\.jsdelivr\.net/npm/bootstrap[^>]*>(?:</script>)?\s*'
)


def _strip_bootstrap_cdn(html_path):
    # pyvis's bundled template.html unconditionally references Bootstrap from
    # a CDN for the "card"/"card-body" wrapper div, regardless of
    # cdn_resources (which only inlines vis-network's own JS/CSS). That div
    # is purely cosmetic -- the graph itself is vis-network, already inlined,
    # and has no Bootstrap dependency -- so strip the two tags to make the
    # file genuinely self-contained.
    with open(html_path, encoding="utf-8") as f:
        html = f.read()
    html = _BOOTSTRAP_TAG_RE.sub("", html)
    with open(html_path, "w", encoding="utf-8") as f:
        f.write(html)


def render_network(graph, out_path, notebook=False):
    net = Network(height="800px", width="100%", notebook=notebook, cdn_resources="in_line")
    net.from_nx(graph)
    net.write_html(out_path, notebook=notebook)
    _strip_bootstrap_cdn(out_path)
    return net


graph = build_bipartite_graph(field_tables, seed_tags)
net = render_network(graph, NETWORK_OUT, notebook=True)
print(f"{graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges -> {NETWORK_OUT}")
net.show(NETWORK_OUT, notebook=True)

## Co-occurrence heatmaps

One heatmap per field: rows = seed tags, columns = attribute values, cell = co-occurrence count (or %% of that seed tag's works if `NORMALIZE = True`). A seed tag with no surviving edges for a field (e.g. filtered out by `MIN_COUNT`) still appears as an all-zero row rather than disappearing.

In [ ]:
def cooccurrence_matrix(counts, field, seed_tags, normalize_by=None):
    if counts.empty:
        return pd.DataFrame(index=seed_tags)
    matrix = counts.pivot(index="tag", columns=field, values="count")
    matrix = matrix.fillna(0)
    columns = sorted(matrix.columns)
    matrix = matrix.reindex(index=seed_tags, columns=columns, fill_value=0)
    if normalize_by is not None:
        matrix = matrix.div(normalize_by.reindex(seed_tags), axis=0) * 100
    else:
        matrix = matrix.astype(int)
    return matrix


def render_heatmap(matrix, field, out_path, normalized=False):
    if matrix.empty or matrix.shape[1] == 0:
        print(f"  skipping heatmap for {field}: no data after filtering")
        return
    width = max(8, 0.35 * matrix.shape[1] + 3)
    height = max(6, 0.22 * matrix.shape[0] + 3)
    fig, ax = plt.subplots(figsize=(width, height))
    annot = matrix.shape[0] * matrix.shape[1] <= 500
    fmt = ".1f" if normalized else "d"
    label = "% of tag's works" if normalized else "co-occurrence count"
    sns.heatmap(matrix, annot=annot, fmt=fmt, cmap="viridis",
                cbar_kws={"label": label}, ax=ax)
    ax.set_xlabel(field)
    ax.set_ylabel("seed tag")
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.show()
    print(f"  wrote {out_path} ({matrix.shape[0]}x{matrix.shape[1]})")


os.makedirs(HEATMAP_OUT_DIR, exist_ok=True)
work_counts = df["tag"].value_counts() if NORMALIZE else None
for field, counts in field_tables.items():
    matrix = cooccurrence_matrix(counts, field, seed_tags, normalize_by=work_counts)
    out_path = os.path.join(HEATMAP_OUT_DIR, f"heatmap_{field}.png")
    render_heatmap(matrix, field, out_path, normalized=NORMALIZE)

## Done

`{NETWORK_OUT}` and `{HEATMAP_OUT_DIR}/heatmap_<field>.png` (one per field in
`FIELDS_TO_VISUALIZE`) are now in the notebook's working directory. Re-run from the
Configuration cell with different `TOP_FANDOMS`/`TOP_ADDITIONAL_TAGS`/`MIN_COUNT`/
`TOP_SEED_TAGS` values to adjust what's included.